# 3. JSONLoader

The loader for **JSON** data (the format most APIs and app configs use). It's a little more advanced
than the others because JSON can be deeply nested — so it uses a small query language called **`jq`**
to pick out exactly the parts you want.

---

## 1. Simple Definition

> **Kid version:** JSON is like a set of **nested boxes** — a box inside a box inside a box. Somewhere
> deep inside is the toy you actually want. `JSONLoader` is a helper you give a **treasure map**
> (`jq_schema`) that says "go into this box, then that box, and grab *these* toys." It then puts each
> toy in a tidy `Document`.

**Professional definition:** `JSONLoader` reads a `.json` (or `.jsonl`) file and extracts content
using a **`jq_schema`** expression, producing one `Document` per matched element.

```python
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="chat.json",
    jq_schema=".messages[].content",   # the "treasure map"
    text_content=True,
)
docs = loader.load()
```

> **Install note:** `JSONLoader` needs the `jq` package: `pip install jq`.

---

## 2. Why Does It Exist?

**The problem:** JSON is **nested and irregular**. You rarely want the *whole* file — you want
specific fields (e.g. every message's `content`, or every product's `description`) from possibly deep
inside the structure.

Example `chat.json`:

```json
{
  "conversation_id": "abc",
  "messages": [
    {"role": "user", "content": "Hello"},
    {"role": "ai",   "content": "Hi there!"}
  ]
}
```

### Before LangChain

```python
import json
data = json.load(open("chat.json"))
texts = [m["content"] for m in data["messages"]]
# You hand-navigate the structure, then hand-build Documents + metadata. Repetitive.
```

### After LangChain

```python
JSONLoader("chat.json", jq_schema=".messages[].content").load()
# jq_schema describes the path once; each match becomes a Document.
```

The `jq_schema` declaratively describes **where** the content lives, no manual looping — and it works
for any nesting depth.

---

## 3. Real-Life Analogy

A **treasure map with an X** 🗺️. The JSON file is a big island with many buildings and rooms. The
`jq_schema` is the map that says "enter the `messages` building, visit every room `[]`, take the
`content` from each." You don't wander the whole island — you go straight to the treasure.

Or a **highlighter on a form**: JSON is a long multi-page form; `jq_schema` highlights exactly which
fields to copy out.

---

## 4. Where It Fits in LangChain Architecture

```
BaseLoader
    │
    ▼
JSONLoader          ← reads JSON, extracts via jq_schema → one Document per match
```

- **`BaseLoader` → `JSONLoader`:** inherits the standard `.load()` interface; the special part is the
  `jq_schema`-driven extraction.
- Like `CSVLoader`, it typically produces **many** documents (one per matched element).

---

## 5. Internal Working

```
  "chat.json"
        │
        ▼
  PARSE the JSON into a Python object
        │
        ▼
  APPLY the jq_schema  (".messages[].content")
     → matches: "Hello", "Hi there!"
        │
        ▼
  FOR EACH match:
     (optional) content_key / metadata_func pick text + metadata
        │
        ▼  wrap in a Document
     Document(page_content="Hello",     metadata={"source": "chat.json", "seq_num": 1})
     Document(page_content="Hi there!", metadata={"source": "chat.json", "seq_num": 2})
        │
        ▼
  return [ Document, Document ]
```

---

## A 60-second `jq_schema` primer

`jq` is a tiny query language for JSON. The few patterns you'll actually use:

| `jq_schema` | Meaning |
|-------------|---------|
| `.` | the whole document |
| `.name` | the `name` field at the top level |
| `.messages[]` | **each item** in the `messages` array |
| `.messages[].content` | the `content` field of each message |
| `.[]` | each item when the file is a top-level array `[ {...}, {...} ]` |
| `.data.items[].text` | drill down: `data` → `items` (each) → `text` |

---

In [1]:
from pprint import pprint

def pretty_print_doc(doc):
    print("=" * 80)
    print("📄 CONTENT")
    print("-" * 80)
    print(doc.page_content)

    print("\n🏷️ METADATA")
    print("-" * 80)
    pprint(doc.metadata)

    print("=" * 80)
    print()

## 6. Attributes (constructor arguments)

### `file_path`

**Definition:** Path to the `.json` / `.jsonl` file.

### `jq_schema`  *(the important one)*

**Definition:** A `jq` expression selecting which parts of the JSON to extract.

**Why it exists:** JSON is nested. you need a precise way to point at the content you want.

**When developers use it:** Always — it's how you target the right fields.

**Real-life use case:** The treasure map's directions.

### `text_content`

**Definition:** Boolean. If `True` (default), the extracted content must be a string. Set `False` to
allow non-string content (numbers, objects) to be stringified.

**Why it exists:** Guards against accidentally embedding non-text; relax it when your content isn't a
plain string.

In [2]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(file_path = r"knowledge-source\apparels.json",
                    jq_schema=".products[]",
                    text_content=False)

loader

C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_19676\1504120167.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader
d:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
{"productID": "0000001", "manufacturer": "Zara", "img": "https://static.zara.net/photos///2023/I/0/2/p/5320/355/800/2/w/563/5320355800_1_1_1.jpg?ts=1697787915583", "Url": "https://www.zara.com/in/en/man-outerwear-l715.html", "productName": "PINSTRIPE COAT", "Description": "Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.", "price": 4900, "category": "Men Cloths"}

🏷️ METADATA
--------------------------------------------------------------------------------
{'seq_num': 1,
 'source': 'D:\\Dev_Workspace\\LangChain\\langchain_tutorials\\document_loaders\\knowledge-source\\apparels.json'}



### `content_key`

**Definition:** When `jq_schema` points at **objects** (not strings), `content_key` names the field
inside each object to use as `page_content`.

**Why it exists:** Often you want the whole object for metadata, but only one field as the text.

**When developers use it:** `jq_schema=".messages[]"` (objects) + `content_key="content"`.

**Real-life use case:** "Grab each whole card, but the *body text* is the `content` field."

In [4]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(file_path = r"knowledge-source\apparels.json",
                    jq_schema=".products[]",
                    text_content=False,
                    content_key="Description")

loader

In [5]:
docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

🏷️ METADATA
--------------------------------------------------------------------------------
{'seq_num': 1,
 'source': 'D:\\Dev_Workspace\\LangChain\\langchain_tutorials\\document_loaders\\knowledge-source\\apparels.json'}



### `metadata_func`

**Definition:** A function `(record, metadata) -> metadata` that lets you **add custom metadata** from
each JSON record.

**Why it exists:** You'll often want fields like `role`, `timestamp`, or `id` saved as metadata for
filtering and citations.

**When developers use it:** Whenever useful info lives alongside the content.

**Real-life use case:** Writing the sender + time on each message card's label.


In [6]:
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["product_name"] = record["productName"]
    metadata["category"] = record["category"]
    metadata["price"] = record["price"]
    # delete seq num
    del metadata["seq_num"]
    return metadata

In [7]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(file_path = r"knowledge-source\apparels.json",
                    jq_schema=".products[]",
                    text_content=False,
                    content_key="Description",
                    metadata_func=metadata_func)

loader

In [8]:
docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

🏷️ METADATA
--------------------------------------------------------------------------------
{'category': 'Men Cloths',
 'price': 4900,
 'product_name': 'PINSTRIPE COAT',
 'source': 'D:\\Dev_Workspace\\LangChain\\langchain_tutorials\\document_loaders\\knowledge-source\\apparels.json'}



### `json_lines`

**Definition:** Set `True` for **JSON Lines** files (`.jsonl`) — one JSON object per line, not one big
array.

**Why it exists:** Logs, exports, and streaming datasets often use `.jsonl`.